In [1]:
import torch
import json
import re
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from peft import PeftModel
import logging
from dataclasses import dataclass
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class ArgumentUnit:
    """Represents an identified argument unit (ADU)"""
    text: str
    start_pos: int
    end_pos: int
    type: str  # 'claim' or 'premise'
    confidence: float

@dataclass
class StanceRelation:
    """Represents a stance relationship between a claim and premise"""
    claim_text: str
    premise_text: str
    stance: str  # 'pro' or 'con'
    confidence: float

@dataclass
class ArgumentStructure:
    """Complete argument structure for a text"""
    original_text: str
    claims: List[ArgumentUnit]
    premises: List[ArgumentUnit]
    stance_relations: List[StanceRelation]

class ModernBERTArgumentMiner:
    """
    Inference pipeline for ModernBERT argument mining models
    """
    
    def __init__(self, base_model_path: str, adapter_paths: Dict[str, str], device: str = None):
        """
        Initialize the argument mining pipeline
        
        Args:
            base_model_path: Path to the base ModernBERT model
            adapter_paths: Dictionary mapping task names to adapter paths
            device: Device to use for inference
        """
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.base_model_path = base_model_path
        self.adapter_paths = adapter_paths
        
        # Initialize tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_path)
        
        # Initialize models (loaded on demand)
        self.models = {}
        
        # Task configurations
        self.task_configs = {
            'adu_identification': {
                'labels': ['No', 'Yes'],
                'task_type': 'token_classification'
            },
            'adu_classification': {
                'labels': ['claim', 'premise'],
                'task_type': 'sequence_classification'
            },
            'stance_classification': {
                'labels': ['con', 'pro'],
                'task_type': 'sequence_classification'
            }
        }
        
        logger.info("ModernBERT Argument Miner initialized")
    
    def load_model_for_task(self, task_name: str):
        """Load model with specific LoRA adapter for a task"""
        if task_name in self.models:
            return self.models[task_name]
            
        logger.info(f"Loading model for task: {task_name}")
        
        # Load base model
        if self.task_configs[task_name]['task_type'] == 'token_classification':
            base_model = AutoModelForTokenClassification.from_pretrained(
                self.base_model_path,
                num_labels=len(self.task_configs[task_name]['labels'])
            )
        else:
            base_model = AutoModelForSequenceClassification.from_pretrained(
                self.base_model_path,
                num_labels=len(self.task_configs[task_name]['labels'])
            )
        
        # Load LoRA adapter
        model = PeftModel.from_pretrained(base_model, self.adapter_paths[task_name])
        model = model.to(self.device)
        model.eval()
        
        self.models[task_name] = model
        return model
    
    def identify_adus(self, text: str, confidence_threshold: float = 0.5) -> List[ArgumentUnit]:
        """
        Identify argument units in text using token classification
        
        Args:
            text: Input text to analyze
            confidence_threshold: Minimum confidence for ADU identification
            
        Returns:
            List of identified ArgumentUnits
        """
        model = self.load_model_for_task('adu_identification')
        
        # Tokenize
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
            return_offsets_mapping=True
        ).to(self.device)
        
        # Get token to character mapping
        offset_mapping = inputs.pop('offset_mapping')[0]
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            predictions = torch.argmax(logits, dim=-1)
        
        # Convert predictions to spans
        labels = self.task_configs['adu_identification']['labels']
        pred_labels = [labels[p] for p in predictions[0].cpu().numpy()]
        confidences = probabilities[0, :, 1].cpu().numpy()  # Confidence for 'Yes' class
        
        # Extract ADU spans
        adus = []
        current_span = None
        
        for i, (pred_label, confidence, offset) in enumerate(zip(pred_labels, confidences, offset_mapping)):
            if pred_label == 'Yes' and confidence > confidence_threshold:
                if current_span is None:
                    current_span = {
                        'start': offset[0].item(),
                        'end': offset[1].item(),
                        'confidence': confidence
                    }
                else:
                    current_span['end'] = offset[1].item()
                    current_span['confidence'] = max(current_span['confidence'], confidence)
            else:
                if current_span is not None:
                    # End current span
                    span_text = text[current_span['start']:current_span['end']].strip()
                    if span_text:
                        adus.append(ArgumentUnit(
                            text=span_text,
                            start_pos=current_span['start'],
                            end_pos=current_span['end'],
                            type='unknown',  # Will be classified later
                            confidence=float(current_span['confidence'])
                        ))
                    current_span = None
        
        # Handle span at end of text
        if current_span is not None:
            span_text = text[current_span['start']:current_span['end']].strip()
            if span_text:
                adus.append(ArgumentUnit(
                    text=span_text,
                    start_pos=current_span['start'],
                    end_pos=current_span['end'],
                    type='unknown',
                    confidence=float(current_span['confidence'])
                ))
        
        return adus
    
    def classify_adu_type(self, adu_text: str) -> tuple[str, float]:
        """
        Classify ADU as claim or premise
        
        Args:
            adu_text: Text of the ADU to classify
            
        Returns:
            Tuple of (classification, confidence)
        """
        model = self.load_model_for_task('adu_classification')
        
        inputs = self.tokenizer(
            adu_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(self.device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            prediction = torch.argmax(logits, dim=-1)
            confidence = torch.max(probabilities).item()
        
        labels = self.task_configs['adu_classification']['labels']
        return labels[prediction.item()], confidence
    
    def classify_stance(self, claim_text: str, premise_text: str) -> tuple[str, float]:
        """
        Classify stance relationship between claim and premise
        
        Args:
            claim_text: Text of the claim
            premise_text: Text of the premise
            
        Returns:
            Tuple of (stance, confidence)
        """
        model = self.load_model_for_task('stance_classification')
        
        # Combine texts (adjust format based on your training)
        combined_text = f"[CLS] {claim_text} [SEP] {premise_text} [SEP]"
        
        inputs = self.tokenizer(
            combined_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(self.device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            prediction = torch.argmax(logits, dim=-1)
            confidence = torch.max(probabilities).item()
        
        labels = self.task_configs['stance_classification']['labels']
        return labels[prediction.item()], confidence
    
    def extract_sentences(self, text: str) -> List[str]:
        """
        Simple sentence extraction as fallback for ADU identification
        """
        # Simple sentence splitting - you might want to use a more sophisticated approach
        sentences = re.split(r'[.!?]+', text)
        return [s.strip() for s in sentences if s.strip()]
    
    def process_text(self, text: str, use_sentence_fallback: bool = True) -> ArgumentStructure:
        """
        Process raw text and extract complete argument structure
        
        Args:
            text: Raw input text
            use_sentence_fallback: If True, use sentence-level analysis as fallback
            
        Returns:
            Complete ArgumentStructure
        """
        logger.info(f"Processing text of length {len(text)}")
        
        # Step 1: Identify ADUs
        adus = self.identify_adus(text)
        
        # Fallback: If no ADUs identified, use sentence-level analysis
        if not adus and use_sentence_fallback:
            logger.info("No ADUs identified, using sentence-level fallback")
            sentences = self.extract_sentences(text)
            adus = []
            current_pos = 0
            
            for sentence in sentences:
                start_pos = text.find(sentence, current_pos)
                if start_pos != -1:
                    adus.append(ArgumentUnit(
                        text=sentence,
                        start_pos=start_pos,
                        end_pos=start_pos + len(sentence),
                        type='unknown',
                        confidence=0.5  # Default confidence for sentence fallback
                    ))
                    current_pos = start_pos + len(sentence)
        
        # Step 2: Classify ADU types
        claims = []
        premises = []
        
        for adu in adus:
            adu_type, confidence = self.classify_adu_type(adu.text)
            adu.type = adu_type
            adu.confidence = min(adu.confidence, confidence)  # Take minimum confidence
            
            if adu_type == 'claim':
                claims.append(adu)
            else:
                premises.append(adu)
        
        # Step 3: Determine stance relationships
        stance_relations = []
        
        for claim in claims:
            for premise in premises:
                stance, confidence = self.classify_stance(claim.text, premise.text)
                stance_relations.append(StanceRelation(
                    claim_text=claim.text,
                    premise_text=premise.text,
                    stance=stance,
                    confidence=confidence
                ))
        
        return ArgumentStructure(
            original_text=text,
            claims=claims,
            premises=premises,
            stance_relations=stance_relations
        )
    
    def process_json_input(self, json_input: str) -> str:
        """
        Process JSON input and return structured JSON output
        
        Args:
            json_input: JSON string with 'text' field
            
        Returns:
            JSON string with structured argument analysis
        """
        try:
            input_data = json.loads(json_input)
            text = input_data.get('text', '')
            
            if not text:
                raise ValueError("No 'text' field found in input JSON")
            
            # Process the text
            result = self.process_text(text)
            
            # Convert to structured output
            output = {
                'original_text': result.original_text,
                'claims': [
                    {
                        'text': claim.text,
                        'start_position': claim.start_pos,
                        'end_position': claim.end_pos,
                        'confidence': claim.confidence
                    }
                    for claim in result.claims
                ],
                'premises': [
                    {
                        'text': premise.text,
                        'start_position': premise.start_pos,
                        'end_position': premise.end_pos,
                        'confidence': premise.confidence
                    }
                    for premise in result.premises
                ],
                'stance_relations': [
                    {
                        'claim': relation.claim_text,
                        'premise': relation.premise_text,
                        'stance': relation.stance,
                        'confidence': relation.confidence
                    }
                    for relation in result.stance_relations
                ],
                'summary': {
                    'total_claims': len(result.claims),
                    'total_premises': len(result.premises),
                    'pro_relations': len([r for r in result.stance_relations if r.stance == 'pro']),
                    'con_relations': len([r for r in result.stance_relations if r.stance == 'con'])
                }
            }
            
            return json.dumps(output, indent=2)
            
        except Exception as e:
            logger.error(f"Error processing JSON input: {str(e)}")
            error_output = {
                'error': str(e),
                'status': 'failed'
            }
            return json.dumps(error_output, indent=2)

def main():
    """
    Example usage of the inference pipeline
    """
    # Configuration
    BASE_MODEL_PATH = "answerdotai/ModernBERT-base"
    ADAPTER_PATHS = {
        'adu_identification': "./argument-mining-modernbert-all/argument-mining-modernbert-adu_identification",
        'adu_classification': "./argument-mining-modernbert-all/argument-mining-modernbert-adu_classification",
        'stance_classification': "./argument-mining-modernbert-all/argument-mining-modernbert-stance_classification"
    }
    
    # Initialize pipeline
    miner = ModernBERTArgumentMiner(
        base_model_path=BASE_MODEL_PATH,
        adapter_paths=ADAPTER_PATHS
    )
    
    # Example JSON input
    example_input = {
        "text": "Look, I'm not saying social media is entirely bad or anything, but honestly the way people use it nowadays is just ridiculous and it's destroying our society piece by piece. My cousin posted 47 pictures of her breakfast last week - FORTY SEVEN! Who needs to see that many photos of scrambled eggs? But that's not even the worst part, the worst part is how these platforms are literally designed to be addictive, they have teams of psychologists and data scientists working around the clock to keep you scrolling and clicking and liking because every second you spend on their platform equals more money in their pockets. The average person checks their phone 96 times per day according to some study I read somewhere, which is absolutely insane if you think about it - that's like every 10 minutes during waking hours! And don't even get me started on how it affects teenagers, my nephew can't even have a conversation without looking at his phone every thirty seconds, it's like he's physically incapable of focusing on anything for more than a minute. Studies show that heavy social media use is linked to depression, anxiety, and sleep disorders, particularly among young people aged 13-18. But then again, you could argue that social media has revolutionized communication and brought people together in ways we never thought possible - I mean, my grandmother talks to her sister in Poland every day through video calls, and during the pandemic, social media was literally the only way many people could stay connected to their loved ones. Plus, it's given a voice to marginalized communities who were previously ignored by mainstream media, activists can organize protests and raise awareness about important issues with just a few clicks. However, the problem is that the same technology that can spread awareness about climate change can also spread conspiracy theories and misinformation at lightning speed."
    }
    
    # Process the input
    json_input = json.dumps(example_input)
    result = miner.process_json_input(json_input)
    
    print("Input:")
    print(json.dumps(example_input, indent=2))
    print("\nOutput:")
    print(result)

if __name__ == "__main__":
    main()

INFO:__main__:ModernBERT Argument Miner initialized
INFO:__main__:Processing text of length 1915
INFO:__main__:Loading model for task: adu_identification
Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO:__main__:No ADUs identified, using sentence-level fallback
INFO:__main__:Loading model for task: adu_classification
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO:__main__:Loading model for task: stance_classification
Some weights of ModernBertForSequenceClassificatio

Input:
{
  "text": "Look, I'm not saying social media is entirely bad or anything, but honestly the way people use it nowadays is just ridiculous and it's destroying our society piece by piece. My cousin posted 47 pictures of her breakfast last week - FORTY SEVEN! Who needs to see that many photos of scrambled eggs? But that's not even the worst part, the worst part is how these platforms are literally designed to be addictive, they have teams of psychologists and data scientists working around the clock to keep you scrolling and clicking and liking because every second you spend on their platform equals more money in their pockets. The average person checks their phone 96 times per day according to some study I read somewhere, which is absolutely insane if you think about it - that's like every 10 minutes during waking hours! And don't even get me started on how it affects teenagers, my nephew can't even have a conversation without looking at his phone every thirty seconds, it's like 